In [ ]:
from jax import random
from jax import numpy as jnp
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import numpyro
from numpyro import distributions as dist
from numpyro import infer
import arviz as az
from plotly.express.colors import qualitative as qual_colours
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from estival.sampling import tools as esamp

from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import get_spaghetti, get_quant_df_from_spaghetti, get_spagh_df_from_dict, add_recovered_to_spaghetti
from emu_renewal.outputs import get_proc_dens, get_prior_vals, get_prior_result_df
from emu_renewal.plotting import plot_post_prior_comparison, plot_spaghetti_calib_comparison
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
PROJECT_PATH = Path.cwd().resolve()
DATA_PATH = PROJECT_PATH.parent / "data/covid_aus"

# Get cases data
target_data = pd.read_csv(DATA_PATH / "WHO-COVID-19-global-data_21_8_24.csv")
seroprev_data = pd.read_csv(DATA_PATH / "aus_seroprev_data.csv")
aust_data = target_data.loc[target_data["Country"] == "Australia"]
aust_data.index = pd.to_datetime(aust_data["Date_reported"], format="%d/%m/%Y")
aust_cases = aust_data["New_cases"].resample("W-SUN").interpolate(method="linear").fillna(0.0)
aust_deaths = aust_data["New_deaths"]

# Clean seroprevalence data
seroprev_data.index = pd.to_datetime(seroprev_data["date"])
aust_seroprev = seroprev_data["seroprevalence"]

# Hospitalisation data
aust_hosp = pd.read_csv(DATA_PATH / "hosp.csv")
aust_hosp.index = pd.to_datetime(aust_hosp["date"])
aust_hosp = aust_hosp["value"]

# Variant data
aust_vars = pd.read_csv(DATA_PATH / "var.csv", index_col=0)
aust_vars.index = pd.to_datetime(aust_vars.index)

# Mobility
aust_mob = pd.read_csv(DATA_PATH / "aust_mobility.csv", index_col=0)
aust_mob.index = pd.to_datetime(aust_mob.index)
model_mob = aust_mob.mean(axis=1).rolling(7).mean().dropna()

In [ ]:
# Specify fixed parameters and get calibration data
proc_update_freq = 14
init_time = 50
pop = 26e6
analysis_start = datetime(2021, 12, 1)
analysis_end = datetime(2022, 10, 1)
# Start calibration targets slightly late so as not to penalise laggy indicators
data_start = analysis_start + timedelta(14)
init_start = analysis_start - timedelta(init_time)
init_end = analysis_start - timedelta(1)
select_data = aust_cases.loc[data_start: analysis_end]
select_deaths = aust_deaths.loc[data_start: analysis_end]
select_vars = aust_vars.loc[data_start: analysis_end]
hosp_data = aust_hosp[data_start: analysis_end: 7]
init_data = aust_cases.resample("D").asfreq().interpolate().loc[init_start: init_end] / 7.0

In [ ]:
# Define model and fitter
ba2_seed_times = [datetime(2021, 12, 2), datetime(2021, 12, 10)]
ba5_seed_times = [datetime(2022, 2, 15), datetime(2022, 2, 16)]
seed_times = [ba2_seed_times, ba5_seed_times]
proc_fitter = CosineMultiCurve()
strains = ["ba1", "ba2", "ba5"]
model_args = [
    pop, 
    analysis_start, 
    analysis_end, 
    proc_update_freq, 
    proc_fitter, 
    GammaDens(), 
    init_time, 
    init_data, 
    GammaDens(), 
    GammaDens(), 
    strains, 
    "ba1", 
    seed_times, 
    20.0, 
]
no_mob_model = MultiStrainModel(*model_args + [None])
mob_model = MultiStrainModel(*model_args + [model_mob])

In [ ]:
# Define parameter ranges
priors = {
    "gen_mean": dist.TruncatedNormal(7.3, 0.5, low=1.0),
    "gen_sd": dist.TruncatedNormal(3.8, 0.5, low=1.0),
    "cdr": dist.Beta(15, 15), #(16,40)
    "ifr": dist.Beta(3, 200),
    "rt_init": dist.Normal(0.0, 0.25),
    "report_mean": dist.TruncatedNormal(8.0, 0.5, low=1.0),
    "report_sd": dist.TruncatedNormal(3.0, 0.5, low=1.0),
    "death_mean": dist.TruncatedNormal(18.0, 0.5, low=1.0),
    "death_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "admit_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "admit_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "stay_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "stay_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "har": dist.Beta(5, 200),
    "shared_dispersion": dist.HalfNormal(0.5),
    "cross_immunity": 0.5,
}

In [ ]:
# Define calibration and calib data
calib_data = {
    "weekly_cases": StandardDispTarget(select_data, weight=20.0),
    "seropos": StandardDispTarget(aust_seroprev, weight=40.0),
    "weekly_deaths": StandardDispTarget(select_deaths, weight=40.0),
    "occupancy": StandardDispTarget(hosp_data, weight=40.0),
    "prop_ba2": StandardDispTarget(select_vars["ba2"], weight=40.0),
}
no_mob_calib = StandardCalib(no_mob_model, priors, calib_data)
mob_calib = StandardCalib(mob_model, priors, calib_data)

In [ ]:
# Run calibration
no_mob_kernel = infer.NUTS(no_mob_calib.calibration, dense_mass=True, init_strategy=no_mob_calib.custom_init(radius=0.5))
mob_kernel = infer.NUTS(mob_calib.calibration, dense_mass=True, init_strategy=mob_calib.custom_init(radius=0.5))
no_mob_mcmc = infer.MCMC(no_mob_kernel, num_chains=4, num_samples=200, num_warmup=200)
mob_mcmc = infer.MCMC(mob_kernel, num_chains=4, num_samples=200, num_warmup=200)
no_mob_mcmc.run(random.PRNGKey(1))
mob_mcmc.run(random.PRNGKey(1))

In [ ]:
# Grab sample of data from calibrated model outputs
no_mob_idata = az.from_dict(no_mob_mcmc.get_samples(True))
mob_idata = az.from_dict(mob_mcmc.get_samples(True))
no_mob_idata_sampled = az.extract(no_mob_idata, num_samples=20)
mob_idata_sampled = az.extract(mob_idata, num_samples=20)
no_mob_sample_params = esamp.xarray_to_sampleiterator(no_mob_idata_sampled)
mob_sample_params = esamp.xarray_to_sampleiterator(mob_idata_sampled)
no_mob_spaghetti = get_spagh_df_from_dict(get_spaghetti(no_mob_calib, no_mob_sample_params))
mob_spaghetti = get_spagh_df_from_dict(get_spaghetti(mob_calib, mob_sample_params))

In [ ]:
plot_spaghetti_calib_comparison(no_mob_spaghetti, no_mob_calib.targets, list(calib_data.keys()) + ["process"]).update_layout(title="No mobility results")

In [ ]:
plot_spaghetti_calib_comparison(mob_spaghetti, mob_calib.targets, list(calib_data.keys()) + ["process"]).update_layout(title="Mobility results")

## No mobility variable process updates

In [ ]:
az.plot_trace(no_mob_idata, var_names=["proc"]);

## Mobility variable process updates

In [ ]:
az.plot_trace(mob_idata, var_names=["proc"]);

In [ ]:
analysis_names = ["mob", "no_mob"]
analyses = [get_prior_vals(mob_idata), get_prior_vals(no_mob_idata)]
result_df = get_prior_result_df(analysis_names, analyses)

In [ ]:
result_df